# Notebook 04: Confidence as Failure Predictor

**Question:** Can the INT8 model's own confidence score predict when it will disagree with the full model?

**Method:** For each confidence threshold, compute what % of disagreements would be caught by flagging inputs below that threshold. Find the optimal threshold that catches the most failures while flagging the fewest inputs.

**Practical implication:** If confidence predicts failure, you can build a routing system — low-confidence inputs go to the full model, high-confidence inputs use INT8. This recovers most of the accuracy loss at a fraction of the cost.

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from quantscope import (
    load_predictions,
    compute_failure_detection,
    find_optimal_threshold,
    print_failure_detection_report
)

sns.set_theme(style='whitegrid')

In [ ]:
import os
os.makedirs('../results/charts', exist_ok=True)
print('✅ charts/ folder created')

In [ ]:
data   = load_predictions('../results/all_predictions.json')
labels = data['labels']

print(f'Test set: {len(labels)} samples')

In [ ]:
# Compute failure detection for full vs int8
result = compute_failure_detection(
    labels,
    predictions_ref=data['full']['predictions'],
    predictions_opt=data['onnx_int8']['predictions'],
    confidences_opt=data['onnx_int8']['confidences'],
)

print_failure_detection_report(result)

In [ ]:
# Find optimal threshold
optimal = find_optimal_threshold(result, min_recall=0.70)

if optimal:
    print(f'Optimal threshold : {optimal["threshold"]}')
    print(f'Disagreements caught : {optimal["disagreements_caught_pct"]}%')
    print(f'Inputs flagged       : {optimal["inputs_flagged_pct"]}%')
    print(f'Precision            : {optimal["precision_pct"]}%')
else:
    print('No threshold achieves 70% recall — confidence is a weak predictor for this model.')

In [ ]:
# Chart 3: Failure detection curve
thresholds   = [t['threshold'] for t in result['thresholds']]
flagged_pct  = [t['inputs_flagged_pct'] for t in result['thresholds']]
caught_pct   = [t['disagreements_caught_pct'] for t in result['thresholds']]
precision    = [t['precision_pct'] for t in result['thresholds']]

fig, ax1 = plt.subplots(figsize=(9, 5))

ax2 = ax1.twinx()

ax1.plot(thresholds, caught_pct,  'o-', color='#ef4444', linewidth=2,
         markersize=7, label='Disagreements caught (%)')
ax2.plot(thresholds, flagged_pct, 's--', color='#3b82f6', linewidth=2,
         markersize=7, label='Inputs flagged (%)')

# Mark optimal threshold
if optimal:
    ax1.axvline(x=optimal['threshold'], color='#22c55e', linestyle=':',
                linewidth=2, label=f'Optimal threshold ({optimal["threshold"]})')

ax1.set_xlabel('Confidence Threshold', fontsize=11)
ax1.set_ylabel('Disagreements Caught (%)', color='#ef4444', fontsize=10)
ax2.set_ylabel('Inputs Flagged (%)', color='#3b82f6', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#ef4444')
ax2.tick_params(axis='y', labelcolor='#3b82f6')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center left', fontsize=9)

plt.title('Confidence as Failure Predictor: Full vs INT8', fontsize=13)
plt.tight_layout()
plt.savefig('../results/charts/confidence_failure_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved confidence_failure_curve.png')

In [ ]:
# Save results
with open('../results/failure_detection.json', 'w') as f:
    json.dump({
        'result' : result,
        'optimal': optimal,
    }, f, indent=2)

print('✅ Saved failure_detection.json')